# Convert EfficientNet-Lite0 + Linear Head → LiteRT (.tflite)

**Steps:**
1. Install dependencies
2. Upload the two ONNX files from your local `exports/` folder
3. Convert each to TFLite via onnx2tf
4. Verify both models
5. Download the `.tflite` files

**Upload these files when prompted (from `texture_haptics/exports/`):**
- `efficientnet_lite0.onnx` (~13 MB)
- `linear_head.onnx` (~0.02 MB)

The linear head has feature normalization baked in — Android just calls encoder → head, no manual preprocessing.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# After install, the runtime restarts automatically to clear numpy conflicts.
# Run Cell 2 onwards after restart — do NOT re-run Cell 1.
import os, signal

!pip install -q onnx onnxsim onnx2tf ai-edge-litert

print("Dependencies installed. Restarting runtime...")
os.kill(os.getpid(), signal.SIGKILL)

In [ ]:
# ── Cell 2: Upload ONNX files ────────────────────────────────────────────────
from google.colab import files
import os

print("Upload efficientnet_lite0.onnx and linear_head.onnx:")
uploaded = files.upload()

for name in uploaded:
    size_mb = len(uploaded[name]) / 1e6
    print(f"  {name}: {size_mb:.1f} MB")

assert "efficientnet_lite0.onnx" in uploaded, "Missing efficientnet_lite0.onnx"
assert "linear_head.onnx" in uploaded, "Missing linear_head.onnx"
print("Both files uploaded OK")

In [ ]:
# ── Cell 3: Simplify ONNX graphs ─────────────────────────────────────────────
import onnx
from onnxsim import simplify

for name in ["efficientnet_lite0.onnx", "linear_head.onnx"]:
    model = onnx.load(name)
    sim_model, ok = simplify(model)
    assert ok, f"onnxsim failed on {name}"
    onnx.save(sim_model, name)
    print(f"  {name}  simplified OK")

In [ ]:
# ── Cell 4: Convert EfficientNet-Lite0 to TFLite ─────────────────────────────
# onnx2tf transposes NCHW -> NHWC automatically for TFLite
!onnx2tf \
    -i efficientnet_lite0.onnx \
    -o efficientnet_lite0_tf \
    --non_verbose

import glob
tflite_files = glob.glob("efficientnet_lite0_tf/**/*.tflite", recursive=True)
print("Generated TFLite files:", tflite_files)

In [ ]:
# ── Cell 5: Convert linear head to TFLite ────────────────────────────────────
!onnx2tf \
    -i linear_head.onnx \
    -o linear_head_tf \
    --non_verbose

import glob
tflite_files = glob.glob("linear_head_tf/**/*.tflite", recursive=True)
print("Generated TFLite files:", tflite_files)

In [ ]:
# ── Cell 6: Copy the float32 tflite files to root ────────────────────────────
import glob, shutil, os

def pick_float_tflite(search_dir, dest_name):
    all_files = glob.glob(f"{search_dir}/**/*.tflite", recursive=True)
    # onnx2tf float32 model has no _integer_quant/_full_integer_quant suffix
    float_files = [f for f in all_files if "quant" not in f.lower()]
    chosen = (float_files or all_files)[0]
    shutil.copy(chosen, dest_name)
    size = os.path.getsize(dest_name) / 1e6
    print(f"  {dest_name}  ({size:.1f} MB)  <- {chosen}")
    return dest_name

enc_tflite  = pick_float_tflite("efficientnet_lite0_tf", "efficientnet_lite0.tflite")
head_tflite = pick_float_tflite("linear_head_tf",        "linear_head.tflite")

In [ ]:
# ── Cell 7: Verify both models ───────────────────────────────────────────────
import numpy as np
from ai_edge_litert.interpreter import Interpreter

enc = Interpreter(model_path=enc_tflite)
enc.allocate_tensors()
inp_det = enc.get_input_details()[0]
out_det = enc.get_output_details()[0]
print("Encoder:")
print(f"  input  shape={inp_det['shape']}  dtype={inp_det['dtype']}")
print(f"  output shape={out_det['shape']}  dtype={out_det['dtype']}")

# TFLite input is NHWC
dummy_img = np.random.rand(1, 224, 224, 3).astype(np.float32)
enc.set_tensor(inp_det['index'], dummy_img)
enc.invoke()
features = enc.get_tensor(out_det['index'])
assert features.shape == (1, 1280), f"Expected (1,1280) got {features.shape}"
print(f"  OK  features.shape={features.shape}")

head = Interpreter(model_path=head_tflite)
head.allocate_tensors()
inp_det2 = head.get_input_details()[0]
out_det2 = head.get_output_details()[0]
print("\nLinear head (normalization baked in):")
print(f"  input  shape={inp_det2['shape']}  dtype={inp_det2['dtype']}")
print(f"  output shape={out_det2['shape']}  dtype={out_det2['dtype']}")

# Feed raw encoder output — head normalizes internally
head.set_tensor(inp_det2['index'], features)
head.invoke()
dims = head.get_tensor(out_det2['index'])
assert dims.shape == (1, 4), f"Expected (1,4) got {dims.shape}"
print(f"  OK  dims={dims}")
print("\nBoth models verified!")

In [ ]:
# ── Cell 8: Full pipeline smoke-test ─────────────────────────────────────────
# Simulates exactly what Android will do: BGR image -> preprocess -> encoder -> head -> dims
import numpy as np
from PIL import Image
from ai_edge_litert.interpreter import Interpreter

def preprocess_image(img_array_rgb):
    """Resize to 224x224, normalize to [-1, 1] to match EfficientNet-Lite0 training.
    Input:  HxWx3 uint8 RGB
    Output: [1, 224, 224, 3] float32, values in [-1, 1]
    """
    img = Image.fromarray(img_array_rgb).resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - 0.5) / 0.5  # normalize: mean=0.5, std=0.5 per channel
    return arr[np.newaxis]   # add batch dim: [1, 224, 224, 3]

enc  = Interpreter(model_path=enc_tflite)
head = Interpreter(model_path=head_tflite)
enc.allocate_tensors()
head.allocate_tensors()

# Synthetic test image
fake_rgb = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
inp = preprocess_image(fake_rgb)

enc.set_tensor(enc.get_input_details()[0]['index'], inp)
enc.invoke()
features = enc.get_tensor(enc.get_output_details()[0]['index'])

# Head takes raw features — normalization is baked in
head.set_tensor(head.get_input_details()[0]['index'], features)
head.invoke()
dims = head.get_tensor(head.get_output_details()[0]['index'])[0]

print("Full pipeline (synthetic image — random values expected):")
for name, val in zip(["rough", "hard", "friction", "density"], dims):
    bar = '=' * int(val * 20)
    print(f"  {name:10s}: {val:.3f}  |{bar:<20s}|")

In [ ]:
# ── Cell 9: Download TFLite files ────────────────────────────────────────────
from google.colab import files

files.download(enc_tflite)
files.download(head_tflite)

print("Downloaded:")
print(f"  {enc_tflite}")
print(f"  {head_tflite}")
print("Copy both into your Android app's  app/src/main/assets/  folder.")

## Android integration notes

### Assets
Copy both files into `app/src/main/assets/`:
- `efficientnet_lite0.tflite` — image → 1280-dim features
- `linear_head.tflite` — 1280-dim features → [rough, hard, friction, density] in [0,1]

### Image preprocessing (must match training exactly)
```kotlin
// Resize to 224x224, then for each pixel channel:
// float_val = (pixel_uint8 / 255.0f - 0.5f) / 0.5f
// Input tensor layout: [1, 224, 224, 3]  NHWC  float32
```

### Inference pipeline
```
Camera frame
  → (optional) U2-Net crop to object bounding box
  → resize 224×224, normalize [-1,1]     ← preprocess
  → efficientnet_lite0.tflite            ← output: [1, 1280] float32
  → linear_head.tflite                   ← output: [1, 4] float32 in [0,1]
  → map_primitives()                     ← output: 8 haptic primitive weights
```
**No manual feature normalization needed** — it's baked into `linear_head.tflite`.

### build.gradle dependency
```groovy
implementation 'com.google.ai.edge.litert:litert:1.0.1'
```

### Haptic primitive mapping (Java/Kotlin)
Reimplement `map_primitives.py` logic on-device:
```kotlin
// dims = float[4] from linear_head:  [rough, hard, friction, density]
val r=dims[0]; val h=dims[1]; val fr=dims[2]; val d=dims[3]
val raw = mapOf(
    "TICK"       to r*h*d,
    "LOW_TICK"   to r*(1-h)*d,
    "CLICK"      to h*d*(1-r),
    "THUD"       to (1-h)*(1-d),
    "SLOW_RISE"  to (1-r)*(1-h)*fr,
    "QUICK_RISE" to h*(1-r),
    "QUICK_FALL" to (1-fr)*(1-r),
)
val total = raw.values.sum() + 1e-6f
val weights = raw.mapValues { it.value / total * 0.95f }
// SPIN = 0.05f (fixed)
```